# Matrix addition on a 2-D grid

Matrices map naturally onto **2-D grids**: blocks of $16 \times 16$ threads tile the output, and each thread owns one entry. Write `matrix_add(A, B)` returning $C = A + B$.

## Inputs and output
- `A`, `B`: `float32` arrays of shape `(M, N)`, with $M, N \ge 1$ and not necessarily multiples of 16. Row-major (C-contiguous).
- Return `C` as a `float32` array of shape `(M, N)`.

## Launch configuration
- Block: `(16, 16)` threads. Grid: `(ceil(N / 16), ceil(M / 16))` blocks.
- Use `col, row = cuda.grid(2)`. The **x** coordinate (first value) is the one that varies fastest across consecutive threads in a warp, so map it to the **column**: in row-major storage, consecutive columns are consecutive addresses.
- Threads outside the matrix (in the partial blocks along the right and bottom edges) must not write.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

BLOCK = (16, 16)

def matrix_add(A, B):
    """A, B: float32 (M, N). Returns A + B as float32 (M, N)."""
    M, N = A.shape
    d_A, d_B = cuda.to_device(A), cuda.to_device(B)
    d_C = cuda.device_array((M, N), dtype=np.float32)
    grid = (1, 1)  # TODO: enough blocks to cover N columns (x) and M rows (y)
    add_kernel[grid, BLOCK](d_A, d_B, d_C)
    return d_C.copy_to_host()

@cuda.jit
def add_kernel(A, B, C):
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [[10.0, 20.0], [30.0, 40.0]], "dtype": "float32"}], "e": {"__nd__": [[11.0, 22.0], [33.0, 44.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, -1.0, 0.5]], "dtype": "float32"}, {"__nd__": [[0.5, 0.5, 0.5]], "dtype": "float32"}], "e": {"__nd__": [[1.5, -0.5, 1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0341927669942379]], "dtype": "float32"}, {"__nd__": [[1.3597475290298462]], "dtype": "float32"}], "e": {"__nd__": [[1.3939403295516968]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.224721074104309, -0.5103070735931396, -0.2979695200920105, -0.5273842215538025, 0.5697263479232788, -0.056064438074827194, 0.7468855977058411, -1.8473248481750488, 1.5665488243103027, -0.0964321568608284, 0.6803784370422363, -0.13656634092330933, -0.3790985643863678, 0.46311014890670776, 0.8245135545730591, -0.2025298774242401], [-0.15278618037700653, 0.6856986284255981, -0.8703406453132629, -1.5143835544586182, 0.39498186111450195, -0.6705658435821533, -1.9203405380249023, -0.8140536546707153, -0.4675975441932678, -1.1932024955749512, -1.4924638271331787, 0.03663782775402069, 0.8972492814064026, -0.23313207924365997, -0.7435960173606873, 0.3849938213825226], [0.717235803604126, -0.3000105917453766, 0.5446677803993225, 1.0428755283355713, -0.2069564312696457, -0.8135155439376831, 0.34765058755874634, 0.2475457340478897, 1.0988128185272217, -1.2845808267593384, -0.6616129279136658, -0.8381669521331787, -1.734014868736267, 0.1264345496892929, 0.5278041958808899, -0.7387900352478027], [1.3856470584869385, 0.8219243288040161, 0.6273764967918396, 0.40170708298683167, 0.9556695818901062, -1.3319798707962036, 0.6139296293258667, 0.602776825428009, -1.7677185535430908, 0.3470301032066345, -0.25042134523391724, 0.7815226912498474, -0.4390621781349182, -0.018240857869386673, 0.34285151958465576, -0.8762617111206055], [0.5985966324806213, -0.10496319085359573, 0.49248263239860535, -0.5218375325202942, 1.086201548576355, 0.6052019596099854, -0.178025022149086, 0.6319571733474731, 1.2597551345825195, 1.7911754846572876, -1.5735763311386108, 0.8831318020820618, 0.465068519115448, -0.09386078268289566, -1.0066649913787842, 1.2571885585784912], [-1.2617379426956177, 0.566945493221283, 1.301867961883545, -1.599669337272644, -0.302517831325531, -1.309216856956482, 0.24405410885810852, 1.51437509059906, 2.0235559940338135, -1.7781144380569458, -0.5749490261077881, 0.703545093536377, 1.5793726444244385, 0.4212106466293335, -0.7461519837379456, 0.2971315085887909], [-0.016619201749563217, -0.20374071598052979, -0.7344710826873779, 0.38725975155830383, 0.30787962675094604, -0.09298412501811981, -0.2216879427433014, -1.2849165201187134, -0.48617565631866455, 1.2064496278762817, -0.19055825471878052, -1.439697265625, 1.3344428539276123, 0.5302653908729553, 2.1080870628356934, 0.06251216679811478], [-0.4613845646381378, -1.4476447105407715, 1.3238383531570435, 2.5695114135742188, -0.8209349513053894, -0.6470876336097717, 0.5961887836456299, -0.8303454518318176, -0.27055835723876953, -0.34983769059181213, 0.19195888936519623, 1.0948184728622437, 0.022067589685320854, 0.9189100861549377, -0.4198888838291168, 0.3277985453605652], [-2.138225555419922, -1.4499480724334717, 0.7959133982658386, -0.5901494026184082, 0.579914927482605, 0.5423442721366882, 1.3222788572311401, 0.8118590712547302, 1.016991376876831, -0.1116713285446167, -0.6982851624488831, -0.7315587997436523, -0.4880439341068268, -1.1298291683197021, -0.5474435687065125, -0.09256905317306519], [0.25161194801330566, -0.33889079093933105, -1.9236828088760376, -0.07228295505046844, 0.22534579038619995, 1.0844756364822388, 0.577863872051239, -0.6435607075691223, -0.7237765789031982, 2.0105974674224854, 0.7566379308700562, 1.8314398527145386, 2.1294445991516113, -0.818083643913269, 0.38526931405067444, 0.45816701650619507], [0.5596122145652771, 0.5419022440910339, 0.20205463469028473, 0.17412924766540527, -1.5024999380111694, -0.16540119051933289, -0.7476196885108948, 0.12608730792999268, -0.4675462543964386, 0.6185207366943359, 0.8190757036209106, 0.3086923658847809, 0.3161681294441223, 0.09294668585062027, -0.44779789447784424, -0.16450126469135284], [-0.49564605951309204, 0.3879871368408203, 0.014113412238657475, 0.5812911987304688, -1.328648328781128, 0.8877789378166199, -0.7626962661743164, -0.7342821955680847, -0.19744297862052917, -0.5633845925331116, 0.29112520813941956, -0.5741647481918335, -1.0694775581359863, -0.8458027839660645, 1.311983346939087, 0.04425833001732826], [-1.1675302982330322, 0.008377403020858765, -1.5559464693069458, 1.7895665168762207, -1.5244781970977783, 0.4787527024745941, 0.5436256527900696, -1.4500077962875366, 0.300276517868042, 0.9971720576286316, 0.4675752818584442, 0.26101627945899963, 0.9490966796875, 0.16090454161167145, 0.3364252746105194, -0.12671154737472534], [0.6318057179450989, -0.9413841366767883, 0.7917632460594177, -0.5053015351295471, -1.090653419494629, 0.3652489483356476, 1.692955493927002, 0.9616782069206238, -0.5156498551368713, 0.6980596780776978, -0.4542091190814972, -0.12401945888996124, 0.08952849358320236, -2.31721830368042, 0.1916147619485855, -1.029609203338623], [-0.6973986029624939, -1.4743903875350952, -1.5165581703186035, -0.9432644248008728, 0.8255972862243652, 1.666041374206543, -0.025205103680491447, 1.0917909145355225, -0.2639806568622589, -1.9119728803634644, 0.1499166190624237, 0.4456482529640198, -0.42861655354499817, 0.3022242486476898, 0.5725562572479248, -0.863407552242279], [-1.4768340587615967, -0.2212541699409485, -0.2110605388879776, -0.35266321897506714, 0.9872467517852783, 1.7255887985229492, -0.4166470170021057, 0.6988866925239563, 0.9409311413764954, 0.7131564021110535, 1.0474863052368164, -0.3892439603805542, 0.7398063540458679, 2.024482011795044, 0.8036655783653259, -0.6198936700820923]], "dtype": "float32"}, {"__nd__": [[0.6077125072479248, 1.2623802423477173, 0.3689862787723541, -0.5634006857872009, 1.5430805683135986, -1.2497670650482178, 0.5034250617027283, -0.01617557369172573, -1.1936657428741455, 1.2194496393203735, 0.3478876054286957, -1.1667982339859009, 0.6017283797264099, -0.431600421667099, -1.902788519859314, -0.6889051198959351], [0.2635514438152313, 0.635702908039093, 0.1655649095773697, 0.045165009796619415, 0.4699900150299072, -0.21097610890865326, 1.1769546270370483, 0.1486416459083557, 0.293921560049057, 0.5407989621162415, -1.0571688413619995, -0.7092268466949463, 1.5285791158676147, 0.335308700799942, -0.2826160788536072, 0.34148353338241577], [-0.48162224888801575, 0.27856630086898804, -0.6658710837364197, 0.27601882815361023, -1.577702522277832, 1.3309197425842285, -0.5129578709602356, -1.581241250038147, -0.22384169697761536, -0.3709962069988251, 0.15239842236042023, -1.1452505588531494, 0.3974655866622925, 3.623567581176758, -1.278667688369751, 0.3377155661582947], [-0.3107188642024994, 0.1926082968711853, -1.8118090629577637, -1.151009440422058, 0.45965954661369324, -0.03841802105307579, 1.6352381706237793, -0.7047973871231079, 0.1744995266199112, 2.913121461868286, -0.7655614614486694, -0.9038335084915161, -0.035130083560943604, -0.044562678784132004, 0.8441672921180725, 0.12957587838172913], [-0.7567393183708191, 0.21142154932022095, 2.638596773147583, 1.2717736959457397, -2.7821691036224365, -0.1420169174671173, -0.8750905394554138, 0.6222625374794006, -0.17326347529888153, 1.932695746421814, 0.9312355518341064, 0.8904029130935669, 0.18657106161117554, -0.040297653526067734, 0.3401799201965332, 1.2952797412872314], [0.5343897938728333, -0.3511325716972351, 1.2750846147537231, 0.19211037456989288, -0.07761505991220474, -0.7913768291473389, -0.5659922957420349, -0.6360467672348022, -2.7923572063446045, 1.0168461799621582, 0.5664775967597961, -0.21427059173583984, 1.0459837913513184, 0.4379877746105194, 0.3715674877166748, -2.4747636318206787], [-0.23301592469215393, 0.532189130783081, 1.6096868515014648, 1.8402948379516602, 1.4258382320404053, -1.1616095304489136, -2.197671413421631, 0.5704695582389832, 0.027615835890173912, 0.9625532627105713, -0.12588436901569366, 0.21975496411323547, -1.4999890327453613, 0.4480096697807312, -0.44489845633506775, 0.34998536109924316], [0.4594059884548187, -0.1172221451997757, 0.31680113077163696, -0.013607030734419823, 0.7591995000839233, 0.501742959022522, -0.5472163558006287, -0.5879018902778625, 1.7437387704849243, -0.10159549862146378, 0.991807758808136, 0.39338573813438416, -0.5641124844551086, -0.7489737868309021, 0.12012568861246109, 0.25665441155433655], [0.6143637299537659, -1.2669334411621094, -0.5946832895278931, -0.09486620128154755, 1.6007494926452637, -2.3601784706115723, 0.4391116499900818, -0.5400458574295044, 0.4313773512840271, -0.4919789135456085, 0.3645510673522949, 1.5646380186080933, 0.3541964888572693, 0.4333999454975128, 0.3728578984737396, -0.7503135204315186], [0.5430178642272949, -0.20834656059741974, -0.8117136359214783, -0.038547925651073456, -0.22728846967220306, 0.7534797787666321, -0.04283922538161278, -0.5294420719146729, 1.455971598625183, 0.9883877635002136, 1.269729495048523, 0.6076673865318298, -0.10969899594783783, -0.41909340023994446, -0.279339998960495, 0.9476906061172485], [0.11190328001976013, -1.4828431606292725, -0.03908279910683632, -1.4161041975021362, 0.16586583852767944, -1.2038196325302124, -1.263936161994934, 0.1185794547200203, -0.07821432501077652, -1.6617616415023804, 0.7482237219810486, 2.1630966663360596, -1.4813756942749023, 1.8198497295379639, -0.6877894401550293, -0.49409061670303345], [-0.17888721823692322, -1.445277214050293, 1.299127221107483, 0.49269363284111023, -0.7514541745185852, -0.27707716822624207, -1.5163594484329224, -0.3477577269077301, 0.9362131357192993, -1.715289831161499, -0.4362403154373169, 0.1467917412519455, 1.0482393503189087, 1.3493305444717407, -0.45725294947624207, -0.7403068542480469], [0.4503996670246124, -0.3632547855377197, -0.3100193738937378, 1.0801836252212524, 1.6897379159927368, 0.26586756110191345, -0.19221477210521698, 0.5712619423866272, -1.8875157833099365, -0.018249062821269035, -1.0605767965316772, 2.389427900314331, 0.7201999425888062, 0.14136984944343567, -0.19436462223529816, -2.514955997467041], [0.7128044962882996, 1.6293689012527466, -0.7288199067115784, 0.6834045052528381, 0.7225431799888611, -1.5481276512145996, 0.9357403516769409, 1.3547810316085815, 0.22848011553287506, -0.06318199634552002, 0.7879635095596313, -1.540753960609436, 0.09328070282936096, 1.0543419122695923, -1.242435336112976, 0.992571234703064], [-0.4182296693325043, -2.080826997756958, -0.10574795305728912, -0.9718965888023376, 0.3192596435546875, -1.172141194343567, 1.0502759218215942, -0.7388303279876709, 0.35635748505592346, -0.5939799547195435, -0.3229053020477295, 1.1917600631713867, 0.21527844667434692, -0.3950320780277252, -1.2515079975128174, 1.1599513292312622], [0.7328111529350281, -0.32653558254241943, 0.24331101775169373, -1.0295774936676025, -1.0148448944091797, -0.8261877298355103, 0.3534272313117981, 0.2958458960056305, -0.7453029155731201, 0.7377746105194092, 0.7846935391426086, 0.17290815711021423, 1.0383886098861694, -0.2873624563217163, -0.1130211353302002, 0.4728586673736572]], "dtype": "float32"}], "e": {"__nd__": [[1.8324335813522339, 0.7520731687545776, 0.07101675868034363, -1.0907849073410034, 2.112806797027588, -1.3058315515518188, 1.2503106594085693, -1.8635004758834839, 0.3728830814361572, 1.123017430305481, 1.0282660722732544, -1.3033645153045654, 0.22262981534004211, 0.031509727239608765, -1.0782749652862549, -0.8914350271224976], [0.11076526343822479, 1.321401596069336, -0.7047757506370544, -1.4692184925079346, 0.8649718761444092, -0.8815419673919678, -0.743385910987854, -0.6654120087623596, -0.17367598414421082, -0.6524035334587097, -2.5496325492858887, -0.6725890040397644, 2.425828456878662, 0.10217662155628204, -1.0262120962142944, 0.7264773845672607], [0.23561355471611023, -0.02144429087638855, -0.12120330333709717, 1.318894386291504, -1.7846589088439941, 0.5174041986465454, -0.16530728340148926, -1.3336955308914185, 0.8749711513519287, -1.6555770635604858, -0.5092145204544067, -1.9834175109863281, -1.3365492820739746, 3.750002145767212, -0.7508634924888611, -0.40107446908950806], [1.0749281644821167, 1.0145325660705566, -1.1844325065612793, -0.7493023872375488, 1.415329098701477, -1.370397925376892, 2.2491679191589355, -0.10202056169509888, -1.5932190418243408, 3.2601516246795654, -1.0159828662872314, -0.1223108172416687, -0.4741922616958618, -0.06280353665351868, 1.187018871307373, -0.7466858625411987], [-0.15814268589019775, 0.10645835846662521, 3.1310794353485107, 0.7499361634254456, -1.6959675550460815, 0.46318504214286804, -1.0531156063079834, 1.2542197704315186, 1.0864917039871216, 3.7238712310791016, -0.6423407793045044, 1.7735347747802734, 0.6516395807266235, -0.1341584324836731, -0.666485071182251, 2.5524682998657227], [-0.7273481488227844, 0.21581292152404785, 2.5769524574279785, -1.4075589179992676, -0.38013288378715515, -2.1005935668945312, -0.3219381868839264, 0.8783283233642578, -0.768801212310791, -0.7612682580947876, -0.008471429347991943, 0.4892745018005371, 2.625356435775757, 0.8591984510421753, -0.37458449602127075, -2.1776320934295654], [-0.24963513016700745, 0.32844841480255127, 0.8752157688140869, 2.2275545597076416, 1.733717918395996, -1.2545936107635498, -2.4193594455718994, -0.7144469618797302, -0.4585598111152649, 2.1690030097961426, -0.316442608833313, -1.219942331314087, -0.16554617881774902, 0.9782750606536865, 1.6631885766983032, 0.41249752044677734], [-0.001978576183319092, -1.5648669004440308, 1.6406395435333252, 2.5559043884277344, -0.061735451221466064, -0.14534467458724976, 0.04897242784500122, -1.4182473421096802, 1.4731804132461548, -0.4514331817626953, 1.1837666034698486, 1.4882042407989502, -0.5420448780059814, 0.16993629932403564, -0.29976320266723633, 0.5844529867172241], [-1.5238618850708008, -2.716881513595581, 0.20123010873794556, -0.6850156188011169, 2.180664539337158, -1.8178341388702393, 1.7613904476165771, 0.27181321382522583, 1.448368787765503, -0.6036502122879028, -0.33373409509658813, 0.8330792188644409, -0.1338474452495575, -0.6964292526245117, -0.17458567023277283, -0.8428825736045837], [0.7946298122406006, -0.5472373366355896, -2.735396385192871, -0.1108308807015419, -0.0019426792860031128, 1.8379554748535156, 0.5350246429443359, -1.1730027198791504, 0.7321950197219849, 2.9989852905273438, 2.026367425918579, 2.4391071796417236, 2.0197455883026123, -1.2371770143508911, 0.10592931509017944, 1.4058575630187988], [0.6715154647827148, -0.9409409165382385, 0.1629718393087387, -1.241974949836731, -1.3366341590881348, -1.3692208528518677, -2.0115559101104736, 0.24466675519943237, -0.5457605719566345, -1.0432409048080444, 1.5672993659973145, 2.4717891216278076, -1.1652076244354248, 1.9127963781356812, -1.1355873346328735, -0.6585918664932251], [-0.6745332479476929, -1.0572900772094727, 1.313240647315979, 1.0739848613739014, -2.0801024436950684, 0.6107017993927002, -2.279055595397949, -1.0820399522781372, 0.7387701272964478, -2.278674364089966, -0.14511510729789734, -0.4273729920387268, -0.021238207817077637, 0.5035277605056763, 0.8547303676605225, -0.6960484981536865], [-0.7171306610107422, -0.35487738251686096, -1.8659658432006836, 2.8697500228881836, 0.1652597188949585, 0.7446202635765076, 0.3514108657836914, -0.8787458539009094, -1.5872392654418945, 0.9789230227470398, -0.5930014848709106, 2.650444269180298, 1.6692966222763062, 0.3022744059562683, 0.14206065237522125, -2.641667604446411], [1.3446102142333984, 0.6879847645759583, 0.06294333934783936, 0.17810297012329102, -0.3681102395057678, -1.1828787326812744, 2.6286959648132324, 2.3164591789245605, -0.28716975450515747, 0.6348776817321777, 0.33375439047813416, -1.6647734642028809, 0.18280920386314392, -1.2628763914108276, -1.0508205890655518, -0.03703796863555908], [-1.1156282424926758, -3.5552172660827637, -1.6223061084747314, -1.9151610136032104, 1.1448569297790527, 0.4939001798629761, 1.0250707864761353, 0.35296058654785156, 0.09237682819366455, -2.505952835083008, -0.1729886829853058, 1.6374082565307617, -0.21333810687065125, -0.0928078293800354, -0.6789517402648926, 0.29654377698898315], [-0.7440229058265686, -0.5477897524833679, 0.032250478863716125, -1.3822407722473145, -0.027598142623901367, 0.899401068687439, -0.06321978569030762, 0.9947326183319092, 0.19562822580337524, 1.4509310722351074, 1.8321797847747803, -0.21633580327033997, 1.7781949043273926, 1.7371195554733276, 0.6906444430351257, -0.14703500270843506]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.03404400870203972, 0.7938991785049438, -1.2526681423187256, 1.425765037536621, -0.7812714576721191, -0.16469785571098328, -1.3775427341461182, 0.6002223491668701, 1.2738513946533203, 1.0710469484329224, 0.8333749771118164, -1.557143211364746, -0.7821680307388306, -0.4625629782676697, -0.9347342848777771, 1.1226952075958252, 0.1844843178987503, -0.024477804079651833, 0.4875357449054718, -0.951619029045105, -0.9083507061004639, -0.8814619779586792, 0.49762094020843506, 0.3415963649749756, 0.5780924558639526, 1.0783830881118774, -1.357300877571106, -0.5604943633079529, 0.14753809571266174, -1.3487430810928345, -0.5321752429008484, 1.0140289068222046, 1.1026440858840942], [1.076706886291504, 0.043017685413360596, -0.21196754276752472, 1.0795246362686157, -0.3726460039615631, 0.33823898434638977, 1.325278878211975, -0.36757078766822815, -0.6105523109436035, -0.009371273219585419, 0.5983791947364807, 0.4824090301990509, 0.15585777163505554, 0.5564201474189758, -1.59959876537323, -0.8592659831047058, 0.16322611272335052, -0.9286615252494812, -1.3416744470596313, -1.2078198194503784, -0.9181638956069946, -0.17847488820552826, -1.0226258039474487, 0.23489883542060852, 1.6962240934371948, -0.14762631058692932, 1.2746988534927368, 0.28843173384666443, -1.1198234558105469, -0.25640958547592163, -0.9818347692489624, 0.7679353952407837, -1.1530953645706177], [1.3225727081298828, 0.5781879425048828, -2.0223100185394287, 0.7843413949012756, 1.5530163049697876, -0.7905237078666687, 0.9852522015571594, -1.7983816862106323, 0.2521566152572632, -1.6613893508911133, 0.48239099979400635, -1.787962794303894, -2.6328701972961426, 0.3960931599140167, 2.2878897190093994, -0.5956695675849915, 0.8301655650138855, 0.11071109771728516, -0.3235047459602356, 0.6153842806816101, 1.642405390739441, -1.2351280450820923, 0.5638969540596008, 0.3451354205608368, -1.42434561252594, 1.614710807800293, 1.9493587017059326, -0.9527719616889954, 1.7379004955291748, 0.05614611878991127, -0.16324834525585175, 1.5034489631652832, 0.22743286192417145], [0.32317590713500977, 1.5176749229431152, 1.288219928741455, 0.8931494355201721, -1.4005498886108398, -1.5723903179168701, 1.1261411905288696, -0.9267195463180542, 0.148483544588089, -1.074072003364563, -0.1836865246295929, -1.2786951065063477, 1.6301740407943726, -1.018912434577942, -0.1219942644238472, -0.1881994754076004, 1.2060978412628174, -1.4198975563049316, -0.03334203362464905, -0.20923426747322083, 0.529925525188446, 0.4720167815685272, -0.4637465476989746, 0.2395859956741333, -0.37463119626045227, 0.4681152403354645, -0.10258016735315323, -0.6555690765380859, -0.027557022869586945, 0.36378395557403564, 2.2658824920654297, -0.49123185873031616, 0.3066580593585968], [1.4630367755889893, -2.0781686305999756, -1.7430381774902344, -1.1886593103408813, -0.11423654109239578, 2.439976692199707, 0.39136695861816406, 1.2080752849578857, -0.2220393568277359, -2.463723659515381, -2.31363844871521, 1.018619179725647, 0.18485954403877258, -1.2984992265701294, 1.1173206567764282, -1.5389549732208252, 1.1342421770095825, 1.4513407945632935, -0.7648233771324158, -1.1081169843673706, 0.8443270325660706, -1.095288634300232, 0.8618864417076111, -1.1952388286590576, 0.8075785636901855, 0.09386694431304932, -0.43277838826179504, -0.8956846594810486, 0.2375747710466385, -1.8926606178283691, -0.2759641706943512, -0.39615416526794434, 0.6783599257469177], [-1.2538203001022339, 1.7134381532669067, -0.3115807771682739, -1.1381675004959106, 1.3860787153244019, 1.6629310846328735, -0.1877720057964325, -0.8910728096961975, 0.0160529688000679, -0.6904117465019226, -0.3120335340499878, 1.5584293603897095, 0.3919009864330292, 0.14539381861686707, 0.3460122048854828, -0.058089807629585266, 0.841567873954773, 0.8792901039123535, -0.36971673369407654, 0.10508839040994644, 2.413607597351074, 0.4215991497039795, -0.18585337698459625, 2.049691677093506, 0.009432554244995117, -1.097301959991455, 0.6745808720588684, -0.899826169013977, -0.49722251296043396, 0.326325386762619, -0.12676692008972168, -0.7634866237640381, -0.5509566068649292], [-0.5933759808540344, 1.1661864519119263, -0.7210001945495605, -0.5722190737724304, 2.0383543968200684, -0.5537643432617188, 1.4894176721572876, 1.3157322406768799, -0.152751624584198, -1.419391393661499, 0.4530141353607178, 0.6726751327514648, 0.967804491519928, 0.07189423590898514, 1.2045575380325317, 1.390358567237854, -1.054382085800171, 1.4294883012771606, 0.1745423525571823, -0.5657440423965454, -0.3186277151107788, 1.8775721788406372, 0.9840299487113953, -1.7890459299087524, 0.08345750719308853, -0.1398499459028244, -0.1755872666835785, -0.5131552219390869, -1.4563014507293701, 0.8616443276405334, -0.9396897554397583, -0.3700411021709442, -0.6023852825164795], [-0.7278238534927368, 0.7432366609573364, 0.38377416133880615, -0.6741616129875183, -0.944810688495636, -0.4694144129753113, 0.9670242071151733, 0.125921368598938, 0.23308369517326355, -0.32816365361213684, -0.25788041949272156, 0.21654373407363892, -1.8434250354766846, 1.612058162689209, -0.787643551826477, -0.5683143734931946, 1.3384970426559448, 0.22243373095989227, 0.04208355396986008, -0.5833193063735962, -1.5611693859100342, 0.18494535982608795, 1.1952613592147827, -0.8996552228927612, 1.3627548217773438, -1.127142071723938, -0.7340111136436462, 0.5058915615081787, 6.522834883071482e-05, 0.690887987613678, -0.7417380809783936, -1.031761646270752, -1.8670626878738403], [-1.2037112712860107, -1.0761077404022217, -1.426965594291687, 0.30846118927001953, -0.6638851761817932, -1.988295078277588, 2.483482599258423, -0.8348536491394043, -1.255509376525879, 1.239788293838501, 1.1654338836669922, -1.1513409614562988, -1.4739818572998047, -1.0000667572021484, 1.129485845565796, -0.31126075983047485, -0.8086978793144226, -0.02837241254746914, 1.2041914463043213, -1.3895503282546997, -0.38380178809165955, -1.383475661277771, 2.1721553802490234, 1.81831955909729, 0.9511620998382568, -0.13340158760547638, 0.6591266393661499, -2.10201358795166, 0.276426762342453, -1.336165189743042, -1.324772834777832, -0.7047259211540222, -0.35021674633026123], [0.5742192268371582, 0.7006787657737732, 0.2626725733280182, -0.9443365931510925, 0.6759867668151855, -0.1209469810128212, 1.18022882938385, -0.9171306490898132, -2.7632250785827637, -0.4767092168331146, 2.151263475418091, 0.4689873456954956, -0.194329172372818, 0.8836362957954407, 0.5031824707984924, 0.4613412916660309, -0.5135456323623657, 0.07617104053497314, 0.2011920064687729, -0.6426617503166199, -1.1187301874160767, 0.13902266323566437, 0.968450129032135, 0.6473042964935303, 0.4939775764942169, 0.5043149590492249, 2.584002733230591, 0.25988373160362244, 1.042038083076477, 1.9571000337600708, 0.053063105791807175, 0.2645300626754761, 1.0440465211868286], [1.6876633167266846, 0.3131053149700165, 0.36301782727241516, 0.09083957970142365, -1.2055991888046265, 1.1176679134368896, -0.2524987459182739, -1.7187526226043701, -0.1441367119550705, -1.7270299196243286, -1.004696011543274, 0.021083984524011612, -0.9169257879257202, 1.0753227472305298, -1.4641393423080444, 0.2691328823566437, 0.5642681121826172, 1.467471718788147, 0.9299717545509338, 0.24120445549488068, 0.08867354691028595, -0.05003194883465767, 1.1474753618240356, -0.647798478603363, 0.5302385687828064, -0.08412697911262512, 0.25039854645729065, -0.2523863911628723, 2.1453747749328613, -0.6467331647872925, 0.2638084888458252, 0.2868878245353699, 0.6192367672920227], [0.4617531895637512, 0.43603435158729553, 0.9934980869293213, -1.4932655096054077, 0.021734239533543587, -1.2216815948486328, 0.24459578096866608, -1.1706072092056274, -0.9478177428245544, -0.048246245831251144, 1.4088457822799683, 0.5899841785430908, 0.30932170152664185, 1.4161463975906372, -0.9322607517242432, -2.0790228843688965, -0.4107004404067993, -0.26495981216430664, 1.2480376958847046, 2.152547836303711, -1.3163338899612427, 2.4909064769744873, 1.5780632495880127, 0.8522765636444092, 0.3818080425262451, 2.4398858547210693, 0.09437491744756699, -0.4598529636859894, 0.15297850966453552, -1.3553317785263062, 0.48598402738571167, -0.09581363946199417, 0.3445990979671478], [0.37423524260520935, -1.0447908639907837, 1.3303216695785522, -1.0006709098815918, 0.9023624062538147, -1.5790756940841675, 0.3858772814273834, -0.8055641651153564, 0.05724462494254112, -0.6367143392562866, -2.1684751510620117, -1.0997687578201294, 0.0837288349866867, 0.6391448378562927, -1.453936219215393, -2.1098473072052, -0.8568490743637085, -0.19603924453258514, -0.6178174614906311, -0.7942455410957336, 0.8660274147987366, 0.9813162684440613, 0.7571935653686523, -0.922544538974762, 0.09168804436922073, -0.17823214828968048, -1.4120382070541382, 0.8979007601737976, 0.10336226224899292, -0.4695143401622772, 1.4329354763031006, 0.7792069911956787, 0.8713791966438293], [1.2153939008712769, 0.7331652641296387, -1.2500243186950684, -0.37364959716796875, 1.0182842016220093, -0.7538830041885376, 0.05186493694782257, 0.4124680161476135, 1.4561649560928345, 1.313318133354187, 0.9128286838531494, -0.17215624451637268, 0.9290069341659546, 0.5420740842819214, -0.1479790210723877, 1.3713910579681396, 0.5081032514572144, -0.7426578402519226, 1.2794140577316284, -0.24214041233062744, 0.6064618229866028, -1.5118073225021362, -0.007087988778948784, -0.9341909289360046, -0.9610256552696228, -1.899191975593567, 2.1544809341430664, -0.139062762260437, 0.24888040125370026, 1.118491768836975, -0.11150328814983368, 0.5236306190490723, -1.3491207361221313], [0.20616845786571503, -0.40428897738456726, 0.510039746761322, 0.1903166025876999, -1.3764606714248657, -1.8428354263305664, -1.0060802698135376, 0.7459077835083008, 0.4336598515510559, 0.4115520417690277, -2.0111770629882812, -2.681530475616455, -0.61628657579422, -0.32557496428489685, -0.5966591835021973, 0.3610243797302246, -0.16507138311862946, -1.1802427768707275, -0.8745474815368652, 0.1481192708015442, 0.07739625126123428, -0.0698157250881195, 2.2715704441070557, -0.2587800920009613, 0.12378431111574173, -0.992669403553009, -0.35602903366088867, 0.5269494652748108, 0.5694980621337891, -0.15533877909183502, 0.3967377245426178, 0.05658234655857086, -1.8801957368850708], [-3.27447509765625, 2.311171531677246, 0.8309165835380554, -0.056763678789138794, -1.7380383014678955, -1.2754769325256348, 1.3559104204177856, -2.788177490234375, -0.3243311643600464, 0.8685643672943115, 0.14323113858699799, 0.7547928094863892, -0.131249338388443, 2.393451690673828, -1.0279959440231323, -1.3622270822525024, 0.27453234791755676, 1.4295806884765625, -0.05589188262820244, 1.0490049123764038, -0.5308372974395752, -1.1852744817733765, 1.3999205827713013, 1.5687130689620972, 1.5331289768218994, -0.4382525384426117, -1.1552072763442993, -1.1693729162216187, -0.22463519871234894, 0.5824378132820129, 0.509171724319458, -1.4344056844711304, 0.19517281651496887], [0.017730241641402245, 0.09012774378061295, -0.023697257041931152, -0.20192024111747742, 0.5932751297950745, 0.12685956060886383, 0.2254108190536499, -0.6907558441162109, 0.39577779173851013, -0.12115202099084854, -2.115161180496216, 0.4836100935935974, -0.2643015682697296, 0.9262818694114685, -0.16209876537322998, -0.09574252367019653, 0.21168993413448334, -0.01185640785843134, -0.44692981243133545, -1.7932004928588867, 1.8426883220672607, -1.5787862539291382, -0.3415219485759735, 0.653138279914856, 0.3800170123577118, 0.5672684907913208, 0.7691897749900818, 1.2633726596832275, 0.2972833514213562, 2.9066426753997803, -1.0134117603302002, 0.22737890481948853, -0.3608529269695282]], "dtype": "float32"}, {"__nd__": [[-0.5451372861862183, 0.28726518154144287, 1.4582406282424927, -0.7102085947990417, -0.7109557390213013, 0.7244888544082642, 0.5439940690994263, 1.9577568769454956, 1.0278799533843994, 0.22959129512310028, -0.3293730318546295, 0.21526280045509338, -0.17521701753139496, -1.3650909662246704, 0.008234765380620956, -0.1751221865415573, -0.2325407862663269, -0.42930877208709717, 1.6900694370269775, 0.6962661743164062, 0.9368627667427063, -0.6076712012290955, -1.022621512413025, -0.2828836441040039, 0.24407178163528442, -0.3453289270401001, -0.18822795152664185, 0.9614195227622986, 0.2992241084575653, -0.9840149283409119, -0.4279992878437042, -0.657539963722229, 0.18093839287757874], [-1.167251706123352, -0.17874735593795776, 0.6926417350769043, 0.028723962604999542, -0.913876473903656, -0.6164140701293945, 0.12435099482536316, 0.48308154940605164, 0.8370703458786011, -0.14521142840385437, -1.3172497749328613, 1.424992561340332, 0.29788631200790405, 0.40607187151908875, 0.30551663041114807, -1.0374903678894043, -0.31930452585220337, 0.2572585344314575, 1.032628059387207, 0.06018184870481491, -3.119948387145996, -0.24135127663612366, 0.3526540994644165, 1.2938891649246216, 0.7113162279129028, 2.490208864212036, 1.4243422746658325, 1.3476637601852417, 0.9427873492240906, -1.3824353218078613, 0.8470504879951477, 0.5994841456413269, -0.5260205268859863], [0.5866459608078003, 0.418066143989563, -0.4650195837020874, -0.1233912855386734, -0.2632948160171509, 0.23047909140586853, -0.4245753884315491, 0.5449131727218628, -1.9175878763198853, -0.2819282114505768, -1.6846873760223389, -0.48289939761161804, 0.5169365406036377, 0.02161206305027008, -0.11063522100448608, 0.06879237294197083, 1.5695009231567383, -1.1062519550323486, 0.8877331614494324, -1.512596845626831, -0.46744096279144287, 0.1635974645614624, -0.8516802787780762, 0.20513737201690674, 0.7286924123764038, -1.1937426328659058, -0.07737187296152115, -0.37656623125076294, -1.940383791923523, -0.18707415461540222, -0.7977706789970398, 1.3390848636627197, -0.8755627274513245], [1.699472188949585, -0.5334467887878418, 1.9381440877914429, 0.5393889546394348, 0.45384663343429565, 0.8057140111923218, 0.9023076891899109, 1.4032914638519287, 1.732708215713501, -0.34671372175216675, -1.250134825706482, -2.6717562675476074, -0.3872530460357666, 0.7602483034133911, 0.3783622682094574, 0.11518598347902298, -0.3501964807510376, -0.6399925351142883, -0.28367435932159424, 0.38897600769996643, -1.764607310295105, 0.7289218306541443, -1.4584585428237915, -0.15825042128562927, -0.6345183253288269, -0.36952999234199524, -1.0477591753005981, -1.1557211875915527, -0.23337389528751373, -1.0880324840545654, 2.455268383026123, -0.42103084921836853, -0.6616194248199463], [-0.5554501414299011, -1.0777614116668701, -2.4627764225006104, -0.2625177502632141, 0.3036781847476959, -1.510738730430603, -0.001508429297246039, -1.2168664932250977, -0.042957235127687454, 0.35191571712493896, 0.47802793979644775, 0.4276074767112732, -0.7118833661079407, 0.8723000288009644, -0.4559962749481201, 0.34194356203079224, 0.6964702606201172, -0.312386691570282, -1.2928166389465332, 0.6050906777381897, -1.1256983280181885, -0.8996486663818359, -1.1112617254257202, 0.8338878154754639, -0.17202097177505493, -2.2575387954711914, 0.7259497046470642, 2.0618932247161865, 1.7909938097000122, -0.5771152973175049, 0.06803864985704422, 0.4993881285190582, 0.8838751912117004], [0.025115592405200005, -1.1043341159820557, 0.744757354259491, 0.6051027178764343, -1.3408242464065552, -0.37759608030319214, 0.2584249675273895, -0.01353703998029232, 0.5134997367858887, 0.7845667004585266, 0.5305721759796143, -0.12614406645298004, -1.360428810119629, 0.6042048931121826, 1.3895822763442993, 1.963908314704895, -0.04524628445506096, -0.047808870673179626, 1.4342803955078125, 1.4265427589416504, 0.328092485666275, 0.7000026106834412, -0.7743433713912964, 0.5709987282752991, -0.8394697904586792, -0.15864133834838867, -0.13671644032001495, -0.48252010345458984, -2.125680923461914, -0.8402491807937622, 0.31922248005867004, -0.07493984699249268, -0.40779978036880493], [0.4145791828632355, 0.6745865345001221, 0.833379328250885, -0.8218864798545837, 0.0062340619042515755, -0.34973374009132385, -0.8942205309867859, -1.5127911567687988, -0.1146039143204689, -0.8893850445747375, 2.136916399002075, -0.11523324251174927, -0.24113188683986664, -0.414314329624176, 1.8589040040969849, -0.9585285186767578, -0.17768041789531708, -0.7863460779190063, -0.1413620412349701, -0.052498288452625275, 1.0822008848190308, -1.4329453706741333, -0.03763282299041748, 1.5996841192245483, -0.5265783667564392, -0.19427302479743958, -0.9037775993347168, -2.635105609893799, -0.12144815921783447, 1.158819317817688, -0.4846680462360382, -1.8634400367736816, 0.2495400607585907], [0.062128566205501556, 0.9835209846496582, 0.8935942649841309, -0.2871144115924835, 0.3829365670681, -1.816307544708252, 1.0968624353408813, 0.16897082328796387, 1.068286418914795, 1.0457680225372314, -0.3043455183506012, -0.1664341241121292, -0.8917707204818726, -2.0833544731140137, -0.14897193014621735, 0.1366451382637024, -1.102806806564331, 0.6948064565658569, 0.08146980404853821, 0.4366171956062317, -0.7866363525390625, -1.7815136909484863, 0.7609203457832336, 0.5940052270889282, -0.12680695950984955, -0.4381064474582672, -0.29401910305023193, -0.15393824875354767, -0.5258229374885559, -2.0279629230499268, 0.767268180847168, -1.5117566585540771, -0.3498064875602722], [-0.32361114025115967, 0.24438485503196716, -1.6300729513168335, 1.1908091306686401, 0.6398501992225647, 0.7547038793563843, 1.2526731491088867, -0.8122609257698059, -3.148609161376953, 0.32025671005249023, 1.3008010387420654, 1.1516057252883911, -0.5454813241958618, -0.10072936117649078, 0.2772962749004364, -1.8154453039169312, -0.6744498610496521, -0.6432456970214844, 1.5771996974945068, -0.6571857929229736, 1.5657507181167603, 0.009061642922461033, -1.6804941892623901, 2.21447491645813, -0.9744051098823547, 0.6167340278625488, 1.7055870294570923, -0.5351468920707703, -1.230575442314148, 1.0409207344055176, 0.0654173195362091, -1.2273327112197876, -1.0812016725540161], [-0.38966137170791626, 0.9640197157859802, -0.287058025598526, 0.6545200943946838, 1.6527138948440552, 1.2694250345230103, 1.4375786781311035, 1.2319056987762451, -1.6085270643234253, 1.2229435443878174, 0.328540563583374, -0.12439786642789841, -1.0846490859985352, -1.5890077352523804, -1.1110998392105103, -0.021363994106650352, -1.1750879287719727, -0.8164747953414917, -0.6167085766792297, 0.666538655757904, -0.2951798141002655, -0.14589427411556244, 1.3262298107147217, -0.3212429881095886, 0.5992422699928284, -1.057297706604004, 0.3291463553905487, 0.11951477825641632, 0.1753423810005188, -0.46637535095214844, -1.0305984020233154, -0.29952922463417053, -0.46797409653663635], [1.0333753824234009, 0.9078344106674194, -1.2021212577819824, 0.6273230910301208, -0.47973155975341797, 0.45892032980918884, -0.4815574884414673, -1.471767544746399, 1.0162675380706787, 0.4522953927516937, -1.2256871461868286, -1.8914121389389038, -0.5342238545417786, -1.181843638420105, -0.3639146387577057, 0.620276689529419, 1.4690848588943481, -0.3568015694618225, 1.3773530721664429, 1.164344310760498, -0.434419721364975, -1.6670231819152832, 0.853590726852417, 0.6104073524475098, 2.354921340942383, 1.4317753314971924, 0.7843567132949829, 1.6907225847244263, -1.0452712774276733, 0.19699925184249878, -0.2695527970790863, 1.7466892004013062, 0.5698139071464539], [-3.0544097423553467, 1.3762550354003906, -0.9003005027770996, 0.7848349213600159, -0.4703943729400635, 0.2542203962802887, 0.645869255065918, 0.20539790391921997, -0.9420003294944763, 1.411979079246521, 0.410418301820755, -0.0028246694710105658, 0.7206611037254333, -0.5203461050987244, 1.901586651802063, 0.24100860953330994, 0.3821572959423065, 0.3641059398651123, -0.8954163193702698, -1.1567473411560059, 0.23513779044151306, 0.37067344784736633, 1.364582896232605, 0.4700598418712616, -1.0703065395355225, 0.5454326272010803, 0.8466376662254333, 1.0223782062530518, 0.3530128598213196, -1.2022851705551147, 1.180702805519104, -0.7126451730728149, 0.5930529236793518], [-1.5537567138671875, 0.7839295268058777, -0.7658334374427795, 1.1015574932098389, 1.7831124067306519, 1.1884467601776123, -1.149410605430603, -0.9258216619491577, -0.8526060581207275, 0.6804380416870117, 0.36320826411247253, 0.17588333785533905, 0.326613187789917, 0.5967998504638672, 0.6682053804397583, 1.7062327861785889, -0.03995601087808609, 0.2191961109638214, -1.840883731842041, -1.3093453645706177, -0.6796088814735413, -1.1731340885162354, 0.23316039144992828, 1.0529567003250122, -0.42463311553001404, -0.4754325747489929, -0.5024564266204834, 0.13940194249153137, -0.5370442271232605, 0.4339412450790405, 0.3075566291809082, 0.3503631353378296, 0.7694076895713806], [0.03608632832765579, 0.1478251814842224, 1.763002872467041, -0.04284965991973877, 0.8003554344177246, 0.0557023249566555, -0.4571846127510071, -0.4568087160587311, -0.8204005360603333, 0.01135068479925394, -0.6048055291175842, -0.4240492284297943, 1.2183550596237183, -0.13467800617218018, 1.1389453411102295, 0.2641766369342804, 0.8604647517204285, -0.6125485301017761, 0.1453496664762497, 2.6334357261657715, 0.8249422311782837, 1.0005831718444824, 0.23006220161914825, 0.8538740277290344, -1.1662077903747559, -0.8554574847221375, 0.4239656925201416, 0.8148446083068848, -0.4157570004463196, -0.19367210566997528, -0.36844512820243835, -0.1950550079345703, -1.2083535194396973], [-1.4414089918136597, -0.0911095142364502, -1.7477856874465942, -0.9894063472747803, -0.6719051599502563, -1.0654548406600952, 0.22459286451339722, -0.46078431606292725, -0.09674201905727386, -1.0273183584213257, -0.760364294052124, 0.492911159992218, 0.34260880947113037, -0.4866865575313568, 0.3412599563598633, 0.16409029066562653, 0.1349274218082428, 1.569358229637146, -0.42708149552345276, 0.3923336863517761, 0.39361318945884705, -0.26269736886024475, -1.1942617893218994, 0.6606037616729736, 0.23064221441745758, 0.01739664375782013, -0.01002294197678566, -0.013103215023875237, -0.9295695424079895, 0.25536736845970154, -1.8410224914550781, -0.06546352803707123, -1.536941647529602], [-0.3940461277961731, 0.6720966100692749, -0.303653359413147, -0.5251619815826416, 0.4669821262359619, -0.31693708896636963, -0.45648327469825745, -0.4427356421947479, 1.461609959602356, 0.613209068775177, 0.42897719144821167, -1.2427889108657837, 0.5656417608261108, -1.347304344177246, 0.4241904020309448, -1.171822190284729, 1.8956037759780884, -1.7412807941436768, 1.1635712385177612, 0.742286205291748, -1.8344192504882812, -0.3003213703632355, 1.3842202425003052, 1.0756069421768188, -1.7189749479293823, -0.580950915813446, -1.2348157167434692, -1.0861672163009644, -0.7039632797241211, 1.778875708580017, 0.07068753987550735, -0.2887628674507141, -0.39789751172065735], [0.6026076078414917, 0.6273232698440552, -0.9829012155532837, 1.403199553489685, -1.1468315124511719, -0.3413524925708771, 0.3860474228858948, 1.6459484100341797, -0.41080230474472046, 0.22112317383289337, 1.237500548362732, 0.5583157539367676, 1.0620877742767334, 0.973156213760376, 0.1948288083076477, 0.5656929016113281, 0.8514796495437622, -0.09284898638725281, 1.5330637693405151, -2.050992727279663, -0.9856200218200684, 1.3786793947219849, 0.3619896471500397, -0.7088592052459717, 0.31323063373565674, 1.0218591690063477, -0.6235057711601257, 0.7033435702323914, -0.18653243780136108, -0.17770589888095856, -0.1334155946969986, 1.0473265647888184, -1.1835182905197144]], "dtype": "float32"}], "e": {"__nd__": [[-0.5791813135147095, 1.0811643600463867, 0.2055724859237671, 0.7155564427375793, -1.4922271966934204, 0.5597909688949585, -0.8335486650466919, 2.557979106903076, 2.3017313480377197, 1.300638198852539, 0.5040019750595093, -1.341880440711975, -0.9573850631713867, -1.8276538848876953, -0.9264995455741882, 0.9475730061531067, -0.0480564683675766, -0.45378658175468445, 2.177605152130127, -0.25535285472869873, 0.02851206064224243, -1.4891331195831299, -0.5250005722045898, 0.05871272087097168, 0.8221642374992371, 0.7330541610717773, -1.5455288887023926, 0.4009251594543457, 0.44676220417022705, -2.3327579498291016, -0.960174560546875, 0.3564889430999756, 1.2835824489593506], [-0.09054481983184814, -0.13572967052459717, 0.48067420721054077, 1.1082485914230347, -1.2865225076675415, -0.27817508578300476, 1.4496299028396606, 0.11551076173782349, 0.22651803493499756, -0.15458270907402039, -0.7188705801963806, 1.9074015617370605, 0.4537440836429596, 0.9624919891357422, -1.2940821647644043, -1.8967564105987549, -0.15607841312885284, -0.6714029908180237, -0.3090463876724243, -1.147637963294983, -4.038112163543701, -0.4198261499404907, -0.6699717044830322, 1.5287879705429077, 2.4075403213500977, 2.3425824642181396, 2.6990411281585693, 1.6360955238342285, -0.1770361065864563, -1.6388449668884277, -0.1347842812538147, 1.3674194812774658, -1.679115891456604], [1.909218668937683, 0.9962540864944458, -2.4873294830322266, 0.6609501242637634, 1.2897214889526367, -0.5600446462631226, 0.5606768131256104, -1.2534685134887695, -1.665431261062622, -1.9433175325393677, -1.2022963762283325, -2.270862102508545, -2.115933656692505, 0.417705237865448, 2.1772544384002686, -0.5268771648406982, 2.3996665477752686, -0.9955408573150635, 0.5642284154891968, -0.897212564945221, 1.174964427947998, -1.0715305805206299, -0.28778332471847534, 0.5502728223800659, -0.6956532001495361, 0.4209681749343872, 1.8719868659973145, -1.3293381929397583, -0.20248329639434814, -0.13092803955078125, -0.9610190391540527, 2.842533826828003, -0.6481298804283142], [2.0226480960845947, 0.9842281341552734, 3.2263641357421875, 1.432538390159607, -0.9467032551765442, -0.7666763067245483, 2.0284488201141357, 0.4765719175338745, 1.8811917304992676, -1.420785665512085, -1.4338213205337524, -3.950451374053955, 1.242920994758606, -0.2586641311645508, 0.2563680112361908, -0.07301349192857742, 0.8559013605117798, -2.059890031814575, -0.3170163929462433, 0.1797417402267456, -1.2346818447113037, 1.2009385824203491, -1.9222050905227661, 0.08133557438850403, -1.0091495513916016, 0.09858524799346924, -1.1503393650054932, -1.8112902641296387, -0.2609309256076813, -0.7242485284805298, 4.721150875091553, -0.9122626781463623, -0.3549613654613495], [0.9075866341590881, -3.1559300422668457, -4.205814361572266, -1.4511771202087402, 0.18944165110588074, 0.929237961769104, 0.38985854387283325, -0.008791208267211914, -0.26499658823013306, -2.1118078231811523, -1.8356105089187622, 1.4462265968322754, -0.5270237922668457, -0.42619919776916504, 0.6613243818283081, -1.1970114707946777, 1.8307124376296997, 1.1389541625976562, -2.0576400756835938, -0.5030263066291809, -0.2813712954521179, -1.9949373006820679, -0.24937528371810913, -0.36135101318359375, 0.6355575919151306, -2.1636719703674316, 0.29317131638526917, 1.1662085056304932, 2.0285685062408447, -2.469775915145874, -0.20792552828788757, 0.10323396325111389, 1.5622351169586182], [-1.2287046909332275, 0.6091040372848511, 0.43317657709121704, -0.5330647826194763, 0.04525446891784668, 1.2853350639343262, 0.07065296173095703, -0.9046098589897156, 0.529552698135376, 0.094154953956604, 0.21853864192962646, 1.4322853088378906, -0.9685277938842773, 0.7495987415313721, 1.7355945110321045, 1.9058184623718262, 0.7963215708732605, 0.8314812183380127, 1.0645636320114136, 1.5316311120986938, 2.7417001724243164, 1.1216018199920654, -0.9601967334747314, 2.62069034576416, -0.8300372362136841, -1.2559432983398438, 0.5378644466400146, -1.382346272468567, -2.622903347015381, -0.5139237642288208, 0.19245555996894836, -0.8384264707565308, -0.9587563872337341], [-0.17879679799079895, 1.8407729864120483, 0.11237913370132446, -1.3941055536270142, 2.044588565826416, -0.9034980535507202, 0.5951971411705017, -0.19705891609191895, -0.2673555314540863, -2.308776378631592, 2.589930534362793, 0.5574418902397156, 0.7266725897789001, -0.3424201011657715, 3.0634615421295166, 0.4318300485610962, -1.2320624589920044, 0.6431422233581543, 0.03318031132221222, -0.6182423233985901, 0.763573169708252, 0.4446268081665039, 0.9463971257209778, -0.1893618106842041, -0.4431208670139313, -0.3341229557991028, -1.0793648958206177, -3.1482608318328857, -1.5777496099472046, 2.020463705062866, -1.4243577718734741, -2.2334811687469482, -0.3528452217578888], [-0.665695309638977, 1.7267576456069946, 1.277368426322937, -0.9612760543823242, -0.5618741512298584, -2.285722017288208, 2.0638866424560547, 0.29489219188690186, 1.3013701438903809, 0.717604398727417, -0.5622259378433228, 0.050109609961509705, -2.7351956367492676, -0.4712963104248047, -0.9366154670715332, -0.4316692352294922, 0.23569023609161377, 0.9172402024269104, 0.12355335801839828, -0.1467021107673645, -2.3478057384490967, -1.5965683460235596, 1.9561817646026611, -0.305649995803833, 1.235947847366333, -1.5652484893798828, -1.0280301570892334, 0.35195332765579224, -0.5257577300071716, -1.3370749950408936, 0.025530099868774414, -2.543518304824829, -2.2168691158294678], [-1.5273224115371704, -0.8317228555679321, -3.0570385456085205, 1.4992703199386597, -0.024034976959228516, -1.2335911989212036, 3.7361557483673096, -1.6471145153045654, -4.404118537902832, 1.5600450038909912, 2.4662349224090576, 0.00026476383209228516, -2.019463062286377, -1.100796103477478, 1.4067821502685547, -2.126706123352051, -1.4831477403640747, -0.6716181039810181, 2.781391143798828, -2.046736240386963, 1.1819489002227783, -1.3744139671325684, 0.4916611909866333, 4.03279447555542, -0.0232430100440979, 0.48333245515823364, 2.364713668823242, -2.637160539627075, -0.9541486501693726, -0.2952444553375244, -1.2593555450439453, -1.932058572769165, -1.4314184188842773], [0.18455785512924194, 1.6646984815597534, -0.024385452270507812, -0.2898164987564087, 2.328700542449951, 1.1484780311584473, 2.617807388305664, 0.3147750496864319, -4.3717522621154785, 0.7462342977523804, 2.479804039001465, 0.3445894718170166, -1.2789782285690308, -0.7053714394569397, -0.6079173684120178, 0.4399772882461548, -1.6886335611343384, -0.7403037548065186, -0.41551655530929565, 0.02387690544128418, -1.4139100313186646, -0.006871610879898071, 2.294679880142212, 0.32606130838394165, 1.0932198762893677, -0.552982747554779, 2.913149118423462, 0.37939852476119995, 1.2173805236816406, 1.4907246828079224, -0.9775353074073792, -0.03499916195869446, 0.5760724544525146], [2.721038818359375, 1.2209397554397583, -0.8391034603118896, 0.7181626558303833, -1.6853307485580444, 1.5765882730484009, -0.7340562343597412, -3.1905202865600586, 0.872130811214447, -1.2747344970703125, -2.2303831577301025, -1.8703281879425049, -1.4511497020721436, -0.1065208911895752, -1.8280539512634277, 0.8894095420837402, 2.033352851867676, 1.1106700897216797, 2.3073248863220215, 1.4055488109588623, -0.34574615955352783, -1.717055082321167, 2.001066207885742, -0.03739112615585327, 2.885159969329834, 1.3476483821868896, 1.0347552299499512, 1.4383361339569092, 1.100103497505188, -0.4497339129447937, -0.005744308233261108, 2.0335769653320312, 1.1890506744384766], [-2.5926566123962402, 1.8122893571853638, 0.09319758415222168, -0.7084305882453918, -0.44866013526916504, -0.9674612283706665, 0.8904650211334229, -0.9652093052864075, -1.8898180723190308, 1.3637328147888184, 1.8192640542984009, 0.5871595144271851, 1.0299828052520752, 0.8958002924919128, 0.9693259000778198, -1.8380142450332642, -0.028543144464492798, 0.09914612770080566, 0.3526213765144348, 0.9958004951477051, -1.0811960697174072, 2.8615798950195312, 2.942646026611328, 1.3223364353179932, -0.6884984970092773, 2.985318422317505, 0.9410125613212585, 0.5625252723693848, 0.5059913396835327, -2.557616949081421, 1.666686773300171, -0.8084588050842285, 0.9376519918441772], [-1.1795214414596558, -0.260861337184906, 0.5644882321357727, 0.10088658332824707, 2.6854748725891113, -0.3906289339065552, -0.763533353805542, -1.7313858270645142, -0.7953614592552185, 0.0437237024307251, -1.8052668571472168, -0.9238854050636292, 0.4103420376777649, 1.2359447479248047, -0.7857308387756348, -0.40361452102661133, -0.8968051075935364, 0.023156866431236267, -2.4587011337280273, -2.103590965270996, 0.1864185333251953, -0.19181782007217407, 0.9903539419174194, 0.13041216135025024, -0.3329450786113739, -0.6536647081375122, -1.9144946336746216, 1.0373027324676514, -0.4336819648742676, -0.035573095083236694, 1.7404921054840088, 1.1295701265335083, 1.64078688621521], [1.251480221748352, 0.8809904456138611, 0.5129785537719727, -0.4164992570877075, 1.8186396360397339, -0.6981806755065918, -0.4053196907043457, -0.044340699911117554, 0.6357644200325012, 1.3246687650680542, 0.3080231547355652, -0.596205472946167, 2.147361993789673, 0.4073960781097412, 0.9909663200378418, 1.6355676651000977, 1.368567943572998, -1.3552063703536987, 1.4247636795043945, 2.3912954330444336, 1.4314041137695312, -0.5112241506576538, 0.22297421097755432, -0.08031690120697021, -2.1272335052490234, -2.7546494007110596, 2.578446626663208, 0.6757818460464478, -0.16687659919261932, 0.9248196482658386, -0.47994840145111084, 0.32857561111450195, -2.557474136352539], [-1.2352405786514282, -0.49539849162101746, -1.237746000289917, -0.7990897297859192, -2.048365831375122, -2.908290386199951, -0.7814874053001404, 0.28512346744537354, 0.33691781759262085, -0.6157662868499756, -2.7715413570404053, -2.188619375228882, -0.2736777663230896, -0.8122615218162537, -0.255399227142334, 0.5251146554946899, -0.030143961310386658, 0.38911545276641846, -1.3016289472579956, 0.5404529571533203, 0.47100943326950073, -0.33251309394836426, 1.0773086547851562, 0.40182366967201233, 0.3544265329837799, -0.9752727746963501, -0.36605197191238403, 0.5138462781906128, -0.36007148027420044, 0.10002858936786652, -1.4442847967147827, -0.008881181478500366, -3.417137384414673], [-3.6685211658477783, 2.9832682609558105, 0.5272632241249084, -0.581925630569458, -1.2710561752319336, -1.5924140214920044, 0.8994271755218506, -3.2309131622314453, 1.1372787952423096, 1.4817733764648438, 0.5722083449363708, -0.48799610137939453, 0.43439242243766785, 1.046147346496582, -0.6038055419921875, -2.5340492725372314, 2.1701362133026123, -0.31170010566711426, 1.1076793670654297, 1.7912911176681519, -2.3652565479278564, -1.4855958223342896, 2.7841408252716064, 2.644320011138916, -0.1858459711074829, -1.0192034244537354, -2.3900229930877686, -2.255540132522583, -0.9285984635353088, 2.361313581466675, 0.5798592567443848, -1.7231686115264893, -0.20272469520568848], [0.6203378438949585, 0.7174510359764099, -1.0065984725952148, 1.2012792825698853, -0.5535563826560974, -0.21449293196201324, 0.6114582419395447, 0.9551925659179688, -0.015024513006210327, 0.09997115284204483, -0.8776606321334839, 1.0419259071350098, 0.7977862358093262, 1.8994381427764893, 0.032730042934417725, 0.4699503779411316, 1.0631695985794067, -0.10470539331436157, 1.0861339569091797, -3.84419322013855, 0.8570683002471924, -0.20010685920715332, 0.020467698574066162, -0.05572092533111572, 0.6932476758956909, 1.5891276597976685, 0.14568400382995605, 1.9667162895202637, 0.11075091361999512, 2.7289366722106934, -1.1468273401260376, 1.274705410003662, -1.544371247291565]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.43135324120521545, 0.5986196398735046, -1.2826532125473022], [-1.6444165706634521, 0.5052911639213562, -0.06201205402612686], [-0.21064752340316772, 0.07886789739131927, -1.0354903936386108], [2.849175214767456, -1.047393560409546, -0.49863559007644653], [-1.4848307371139526, -0.47916746139526367, 1.462846040725708], [-0.044414304196834564, 0.531455397605896, -0.5467332601547241], [1.7496943473815918, 0.1005396768450737, -1.3116220235824585], [0.7066411375999451, -0.432939738035202, 0.5436133146286011], [0.07581271231174469, 0.2001079022884369, 0.0732012614607811], [-0.9688631892204285, -1.5655864477157593, 0.9122464656829834], [-0.19911935925483704, 0.22192807495594025, -0.5276110768318176], [-2.086988925933838, -1.3408026695251465, 1.3313014507293701], [-0.8494139313697815, -0.44694361090660095, -1.5316957235336304], [-0.5923236608505249, -0.4374735951423645, -0.28680098056793213], [-0.13919708132743835, -0.6972665190696716, -0.6404386162757874], [-0.08013781905174255, 0.25611749291419983, -0.20897582173347473], [1.6754531860351562, 0.13803771138191223, 0.09354804456233978], [-0.7238397598266602, 0.9589059352874756, -1.0919997692108154], [-1.6868062019348145, -1.0442450046539307, 0.5497585535049438], [-1.213990330696106, 1.2240607738494873, -0.09267057478427887], [-1.586313009262085, -0.2994648516178131, 1.7128576040267944], [-0.93077152967453, -1.3806651830673218, 0.441325843334198], [1.2350668907165527, 0.9449986219406128, -0.39992761611938477], [0.11613193899393082, 0.2534978985786438, 1.1024837493896484], [2.0770857334136963, -0.1793394833803177, 1.2276822328567505], [-0.4155627489089966, 0.5173991322517395, -1.0048019886016846], [-0.8356346487998962, 0.34358441829681396, -0.9683771133422852], [0.8154066801071167, -0.6222600340843201, 0.0058388919569551945], [-0.15516750514507294, -0.40503495931625366, 0.05931839719414711], [1.0076042413711548, 3.033585548400879, -1.258474349975586], [0.2285110205411911, -0.24714727699756622, -0.9848824739456177], [0.21473674476146698, -0.9502750039100647, -0.14974260330200195], [1.1681451797485352, -2.092525005340576, 0.17607127130031586], [0.37515872716903687, -1.0289368629455566, 0.45231151580810547], [0.2799926996231079, -0.4205060601234436, 1.1404625177383423], [-1.3789129257202148, 0.5715424418449402, -0.30512985587120056], [-1.1279528141021729, -0.2491394281387329, -0.4208616018295288], [0.7472099661827087, -1.5668308734893799, -0.2662827670574188], [-0.527330756187439, 1.5011065006256104, -1.1899964809417725], [0.9865162968635559, -1.8289299011230469, 0.27081647515296936]], "dtype": "float32"}, {"__nd__": [[-0.209959015250206, 0.30316492915153503, -0.7782979607582092], [-2.0270564556121826, -1.6222025156021118, 1.234379529953003], [0.13416440784931183, -0.9441108703613281, 0.8190341591835022], [-0.8266993761062622, -0.8212763667106628, -1.296074390411377], [1.0936036109924316, 1.877927303314209, -1.0492349863052368], [0.2031872570514679, 0.8908385038375854, 2.0277936458587646], [-2.486335515975952, 0.05198819190263748, 0.5554592609405518], [-1.5702325105667114, 0.3166602849960327, 0.10546060651540756], [0.6309510469436646, -0.6754188537597656, 0.32438087463378906], [-0.6989731192588806, 0.5119256973266602, 0.06392408907413483], [-0.4714347720146179, -1.5103665590286255, 1.552963376045227], [1.0138640403747559, 0.7845959663391113, -0.7597670555114746], [-0.44575363397598267, 1.5747874975204468, 0.6491732001304626], [-0.9468332529067993, -0.04836178198456764, 0.04706668481230736], [-0.29374435544013977, -1.0371133089065552, 1.1618332862854004], [0.49699079990386963, -0.28886160254478455, -1.664081335067749], [-0.6738372445106506, -0.0938907340168953, 0.779319167137146], [-0.4574701189994812, 1.021588683128357, 0.05443437024950981], [-0.8295055031776428, -2.876941442489624, -1.2578526735305786], [-1.5732849836349487, -0.23332637548446655, 0.2516428530216217], [1.310072898864746, -0.0918101966381073, -1.2367613315582275], [2.2747886180877686, 0.2344716340303421, -0.33982834219932556], [-0.33805474638938904, -1.4271225929260254, 0.5615082383155823], [0.300031453371048, 1.5157936811447144, 1.0299919843673706], [0.7971756458282471, 1.1879535913467407, -0.35381343960762024], [-0.5123939514160156, -0.11742368340492249, -1.312925934791565], [-0.20899313688278198, -0.39443162083625793, 0.6049646735191345], [-0.07751093059778214, -1.9703651666641235, -0.03053371235728264], [1.3561124801635742, -1.8946175575256348, -0.8176887631416321], [-0.04048190638422966, -1.3829243183135986, -0.6051079034805298], [0.0009799775434657931, -1.3735504150390625, 0.24097062647342682], [0.356911838054657, -1.3197686672210693, -0.023597823455929756], [1.030198097229004, -0.2245400995016098, 0.38788899779319763], [0.5963340997695923, -0.2808763384819031, 0.9375659823417664], [0.3407047986984253, -1.034389615058899, 0.48002880811691284], [0.823676347732544, 0.7647067308425903, -0.6830645203590393], [0.42164644598960876, 1.335646390914917, 0.19486944377422333], [1.149742841720581, 1.2670501470565796, -0.0690196081995964], [0.6289850473403931, -0.32119712233543396, 1.038358449935913], [-0.11974512785673141, -0.5781483054161072, -0.05319877713918686]], "dtype": "float32"}], "e": {"__nd__": [[-0.6413122415542603, 0.9017845392227173, -2.0609512329101562], [-3.6714730262756348, -1.1169114112854004, 1.1723674535751343], [-0.0764831155538559, -0.8652429580688477, -0.21645623445510864], [2.0224757194519043, -1.8686699867248535, -1.7947099208831787], [-0.391227126121521, 1.3987598419189453, 0.4136110544204712], [0.15877294540405273, 1.4222939014434814, 1.4810603857040405], [-0.7366411685943604, 0.15252786874771118, -0.7561627626419067], [-0.8635913729667664, -0.11627945303916931, 0.6490738987922668], [0.706763744354248, -0.47531095147132874, 0.39758214354515076], [-1.667836308479309, -1.0536607503890991, 0.976170539855957], [-0.6705541610717773, -1.2884384393692017, 1.0253522396087646], [-1.073124885559082, -0.5562067031860352, 0.5715343952178955], [-1.2951675653457642, 1.1278438568115234, -0.8825225234031677], [-1.5391569137573242, -0.48583537340164185, -0.23973429203033447], [-0.4329414367675781, -1.734379768371582, 0.521394670009613], [0.4168529808521271, -0.03274410963058472, -1.8730571269989014], [1.0016160011291504, 0.04414697736501694, 0.872867226600647], [-1.1813099384307861, 1.9804946184158325, -1.0375653505325317], [-2.5163116455078125, -3.9211864471435547, -0.7080941200256348], [-2.7872753143310547, 0.9907343983650208, 0.15897227823734283], [-0.27624011039733887, -0.3912750482559204, 0.4760962724685669], [1.3440170288085938, -1.146193504333496, 0.10149750113487244], [0.8970121145248413, -0.4821239709854126, 0.1615806221961975], [0.4161633849143982, 1.769291639328003, 2.1324758529663086], [2.8742613792419434, 1.0086140632629395, 0.8738688230514526], [-0.9279567003250122, 0.399975448846817, -2.317728042602539], [-1.0446277856826782, -0.05084720253944397, -0.36341243982315063], [0.7378957271575928, -2.592625141143799, -0.02469482086598873], [1.2009450197219849, -2.299652576446533, -0.7583703398704529], [0.9671223163604736, 1.6506612300872803, -1.8635822534561157], [0.2294909954071045, -1.6206977367401123, -0.743911862373352], [0.5716485977172852, -2.2700436115264893, -0.17334042489528656], [2.198343276977539, -2.3170650005340576, 0.5639602541923523], [0.9714928269386292, -1.3098132610321045, 1.3898775577545166], [0.6206974983215332, -1.4548957347869873, 1.6204912662506104], [-0.5552365779876709, 1.3362491130828857, -0.9881943464279175], [-0.7063063383102417, 1.086506962776184, -0.22599215805530548], [1.8969528675079346, -0.2997807264328003, -0.3353023827075958], [0.1016542911529541, 1.179909348487854, -0.15163803100585938], [0.8667711615562439, -2.407078266143799, 0.2176176905632019]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.09951375424861908, 0.3558129072189331, 0.439642071723938, -1.189869999885559, -0.6759539842605591, 0.3360837996006012, 0.45801204442977905, 0.0636146143078804, 2.612687826156616, 1.0993094444274902, -0.094341941177845, 0.7118019461631775, 1.3227680921554565, 0.35164833068847656, -1.7579624652862549, 0.0863533467054367, 0.04128171503543854, 2.124765396118164, -1.8443560600280762, 1.5411441326141357, -0.15991497039794922, 0.9288603663444519, -0.293224036693573, 0.17557393014431, 1.1330533027648926, 0.08690159022808075, 1.0578899383544922, -0.07623197883367538, -0.4042925238609314, -0.21245400607585907, -0.49583494663238525, -0.11223287135362625, 1.3709280490875244, -1.1605408191680908, -1.6501662731170654, -0.007337658200412989, 1.6966227293014526, 1.0595718622207642, 1.3363866806030273, 1.2650772333145142, -0.7137894630432129, 0.13480326533317566, -2.6437032222747803, -0.5915677547454834, 0.626048743724823, -1.537575364112854, -2.0406415462493896, -1.278069019317627, 0.24098990857601166, 1.1913377046585083, 2.6123404502868652, 1.4580212831497192, -1.4869720935821533, -2.289996862411499, 0.1822609305381775, 1.0937267541885376, -1.5820543766021729, -0.48443275690078735, -0.4096463918685913, 0.07658707350492477, -0.135009303689003, 0.5633953213691711, 1.1069068908691406, 0.12307474762201309, 0.8036585450172424, 2.0483503341674805, -0.9651687145233154, -0.9892771244049072, 0.5694360733032227, 1.9142934083938599], [0.23064163327217102, 1.1583210229873657, -0.24123677611351013, 0.9845799207687378, -0.5563954710960388, 0.7466517090797424, 0.46929365396499634, 0.47044456005096436, 1.4546303749084473, 1.1479530334472656, -0.5164524912834167, 0.9699473977088928, -1.612207055091858, 1.0784096717834473, 2.841458320617676, 0.6248390078544617, 0.6046738028526306, 0.1627529114484787, 0.9341251850128174, -0.3463519513607025, 0.7786362171173096, -0.16145941615104675, -0.3116697371006012, 0.23903559148311615, 0.05693879723548889, -1.6170666217803955, 2.196741819381714, 0.26154041290283203, -1.5731934309005737, 1.2009485960006714, 0.09538653492927551, 0.3287743031978607, 0.831670343875885, -0.7703133225440979, -2.818807363510132, 0.025220338255167007, 0.36731308698654175, 1.127985954284668, -0.631602942943573, -0.14142222702503204, 0.5246556997299194, -0.19098037481307983, -0.12410193681716919, 0.3802419900894165, -0.8707066178321838, 1.7093415260314941, -0.7972414493560791, 0.16097679734230042, -0.2048966884613037, -0.5836406946182251, 0.6592050194740295, 1.1348644495010376, 1.5793046951293945, 0.7051250338554382, -0.3255288004875183, 0.9850801229476929, -1.3225916624069214, 0.17032915353775024, 0.6430517435073853, 0.6838953495025635, -1.0335420370101929, -1.3810981512069702, 1.7766578197479248, -0.39810124039649963, -1.5354465246200562, -1.1293165683746338, 0.12187366187572479, 1.2867186069488525, 0.10220125317573547, 1.3037222623825073]], "dtype": "float32"}, {"__nd__": [[0.3278641700744629, 0.5716691017150879, 1.0827052593231201, -0.279274046421051, 0.3914215862751007, -0.2940448522567749, -0.5446316599845886, 0.7473875880241394, -0.7314819097518921, 0.16838142275810242, -0.7041932940483093, -0.003729466814547777, -0.4435728192329407, 1.1763858795166016, 0.15966449677944183, 0.1554154008626938, 1.7243146896362305, -2.82210373878479, -1.0474823713302612, -2.675649881362915, 0.054330937564373016, 0.4971020817756653, 0.7001056671142578, 0.3225477337837219, -0.8200017213821411, 1.8029851913452148, 0.3530975580215454, -0.21469134092330933, -1.076878309249878, 0.46296149492263794, -0.4504893720149994, -0.41084858775138855, 0.24440130591392517, 0.25383704900741577, -0.7863626480102539, 1.286618947982788, -0.9701332449913025, -1.0861010551452637, 0.33748188614845276, 0.15581680834293365, 0.09723509103059769, -0.38538095355033875, 0.06759358942508698, -0.5387552976608276, -1.6251484155654907, 0.7581910490989685, -0.9910189509391785, -0.20723098516464233, 0.7522143721580505, 0.04475650191307068, 2.7624144554138184, -1.379819631576538, -0.7292047142982483, 0.10678300261497498, 1.3744405508041382, -0.2313249260187149, -1.41489839553833, 1.1594443321228027, -0.9062055945396423, -0.1834375411272049, 0.8957656621932983, -0.6558253169059753, -0.43880072236061096, 0.5000855326652527, -0.1919773668050766, 0.297719269990921, -0.9359434247016907, -0.1090845912694931, -1.0825095176696777, -1.548913598060608], [1.7637443542480469, 0.4617643654346466, 0.9235436320304871, -0.9614047408103943, -0.022643115371465683, -0.1997973918914795, -1.4154510498046875, 0.551532506942749, -0.7170168161392212, -0.19801665842533112, -1.5124908685684204, 0.9974308013916016, 0.7635931968688965, 0.39671701192855835, -0.9050182700157166, -0.5346614718437195, 1.2260791063308716, -0.27257323265075684, 0.2710590362548828, 0.14485576748847961, -1.1144986152648926, 1.528316617012024, -1.9798102378845215, 0.22801780700683594, -0.08354317396879196, -1.6138184070587158, 0.33607324957847595, 0.5242025256156921, 0.05571998655796051, -0.8236974477767944, 0.4540460705757141, 1.1187982559204102, 0.22526311874389648, 1.7280607223510742, 0.8633707165718079, 0.7738565802574158, 1.394853115081787, -0.6567584276199341, -0.5165150761604309, 0.5204017758369446, 0.5243023037910461, 1.5143526792526245, 1.1509201526641846, 0.18134093284606934, 0.22525347769260406, -0.2678124010562897, -0.3561781048774719, -0.44199833273887634, 0.9148128628730774, -0.45901742577552795, -0.3419206738471985, 0.17331182956695557, -0.4972454607486725, -0.21193507313728333, -2.164274215698242, 0.39416566491127014, -0.2830442488193512, 0.21655137836933136, 0.395698606967926, -1.824933409690857, 2.4375529289245605, 0.20156878232955933, 0.2297794222831726, -1.7245489358901978, 0.6920809149742126, 0.4913942813873291, -0.27156245708465576, 0.34509071707725525, 1.389080286026001, -0.21988987922668457]], "dtype": "float32"}], "e": {"__nd__": [[0.2283504158258438, 0.927482008934021, 1.522347331047058, -1.4691441059112549, -0.2845323979854584, 0.042038947343826294, -0.08661961555480957, 0.8110021948814392, 1.8812059164047241, 1.267690896987915, -0.7985352277755737, 0.7080724835395813, 0.8791952729225159, 1.5280342102050781, -1.5982979536056519, 0.2417687475681305, 1.7655963897705078, -0.697338342666626, -2.891838550567627, -1.1345057487487793, -0.1055840328335762, 1.4259624481201172, 0.4068816304206848, 0.4981216788291931, 0.31305158138275146, 1.889886736869812, 1.4109874963760376, -0.2909233272075653, -1.481170892715454, 0.2505074739456177, -0.946324348449707, -0.5230814814567566, 1.615329384803772, -0.906703770160675, -2.4365289211273193, 1.2792812585830688, 0.7264894843101501, -0.02652919292449951, 1.6738685369491577, 1.4208940267562866, -0.6165543794631958, -0.2505776882171631, -2.5761096477508545, -1.130323052406311, -0.9990996718406677, -0.7793843150138855, -3.031660556793213, -1.485300064086914, 0.9932042956352234, 1.2360942363739014, 5.374754905700684, 0.07820165157318115, -2.216176748275757, -2.183213949203491, 1.556701421737671, 0.8624018430709839, -2.996952772140503, 0.6750115752220154, -1.3158519268035889, -0.10685046762228012, 0.7607563734054565, -0.0924299955368042, 0.668106198310852, 0.6231603026390076, 0.6116811633110046, 2.346069574356079, -1.9011120796203613, -1.0983617305755615, -0.5130734443664551, 0.36537981033325195], [1.9943859577178955, 1.62008535861969, 0.6823068857192993, 0.023175179958343506, -0.5790385603904724, 0.5468543171882629, -0.9461573958396912, 1.0219770669937134, 0.7376135587692261, 0.9499363899230957, -2.0289433002471924, 1.9673781394958496, -0.8486138582229614, 1.4751267433166504, 1.9364399909973145, 0.09017753601074219, 1.8307528495788574, -0.10982032120227814, 1.2051842212677002, -0.2014961838722229, -0.335862398147583, 1.3668571710586548, -2.29148006439209, 0.4670534133911133, -0.02660437673330307, -3.2308850288391113, 2.5328149795532227, 0.7857429385185242, -1.5174734592437744, 0.37725114822387695, 0.549432635307312, 1.4475725889205933, 1.0569334030151367, 0.9577473998069763, -1.9554367065429688, 0.7990769147872925, 1.7621662616729736, 0.4712275266647339, -1.148118019104004, 0.37897956371307373, 1.0489580631256104, 1.3233723640441895, 1.0268182754516602, 0.5615829229354858, -0.645453155040741, 1.4415291547775269, -1.1534194946289062, -0.2810215353965759, 0.7099161744117737, -1.0426580905914307, 0.31728434562683105, 1.3081762790679932, 1.0820592641830444, 0.4931899607181549, -2.4898030757904053, 1.3792457580566406, -1.6056358814239502, 0.3868805170059204, 1.038750410079956, -1.1410380601882935, 1.4040108919143677, -1.1795294284820557, 2.006437301635742, -2.122650146484375, -0.8433656096458435, -0.6379222869873047, -0.14968879520893097, 1.6318093538284302, 1.491281509399414, 1.0838323831558228]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(matrix_add, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")